In [3]:
import torch
from torch.utils.data import Dataset, DataLoader
import pickle
import numpy as np

# 1. 저장된 전처리 데이터 불러오기
load_path = '../data/processed_data.pkl'

with open(load_path, 'rb') as f:
    saved_data = pickle.load(f)

patient_history = saved_data['patient_history']
vocabs = saved_data['vocabs']

print(f"✅ 데이터 로드 완료!")
print(f"- 환자 수: {len(patient_history)}명")
print(f"- 진단 코드(Vocab) 수: {len(vocabs['diag'])}개")

✅ 데이터 로드 완료!
- 환자 수: 100명
- 진단 코드(Vocab) 수: 1472개


In [4]:
class MIMICDataset(Dataset):
    def __init__(self, patient_history, max_len=50):
        """
        Args:
            patient_history: 전처리된 환자 데이터 딕셔너리
            max_len: 최대 방문 기록 길이 (너무 과거 기록은 자르기 위해)
        """
        self.patient_ids = list(patient_history.keys())
        self.history = patient_history
        self.max_len = max_len

    def __len__(self):
        return len(self.patient_ids)

    def __getitem__(self, idx):
        # 1. 환자 ID 가져오기
        pid = self.patient_ids[idx]
        visits = self.history[pid]
        
        # 2. 데이터 추출 (방문 기록들)
        diag_seq = []
        proc_seq = []
        drug_seq = []
        
        for visit in visits:
            # 코드가 없는 경우를 대비해 빈 리스트 처리
            diag_seq.append(visit.get('conditions', []))
            proc_seq.append(visit.get('procedures', []))
            drug_seq.append(visit.get('drugs', []))
            
        # 3. 길이 조절 (최근 기록 위주로 max_len 만큼만 가져오기)
        # 예: 기록이 100번이면 최근 50번만 사용
        diag_seq = diag_seq[-self.max_len:]
        proc_seq = proc_seq[-self.max_len:]
        drug_seq = drug_seq[-self.max_len:]

        # 4. 결과 반환
        return {
            'patient_id': pid,
            'num_visits': len(diag_seq),
            'diagnosis': diag_seq,   # [[진단1, 진단2], [진단3], ...]
            'procedure': proc_seq,
            'medication': drug_seq
        }

# 데이터셋 인스턴스 생성
dataset = MIMICDataset(patient_history)
print("✅ MIMICDataset 클래스 생성 완료!")

✅ MIMICDataset 클래스 생성 완료!


In [5]:
# 첫 번째 환자 데이터 샘플링
sample = dataset[0]

print(f"=== Sample Data (Patient ID: {sample['patient_id']}) ===")
print(f"총 방문 횟수: {sample['num_visits']}")
print(f"첫 방문 진단 코드: {sample['diagnosis'][0]}")
print(f"첫 방문 시술 코드: {sample['procedure'][0]}")
print(f"첫 방문 약물 코드: {sample['medication'][0]}")

=== Sample Data (Patient ID: 10000032) ===
총 방문 횟수: 4
첫 방문 진단 코드: [409, 533, 516, 108, 211, 410, 403, 977]
첫 방문 시술 코드: [47]
첫 방문 약물 코드: [120, 120, 131, 141, 216, 166, 317, 405, 432, 199, 199, 142, 411, 219]


In [6]:
def collate_fn(batch):
    """
    들쭉날쭉한 환자 데이터를 받아서
    모델에 넣을 수 있는 직사각형 텐서(Tensor)로 변환하는 함수
    """
    # 1. 배치 내에서 가장 긴 방문 기록 길이 찾기
    max_visits = max([sample['num_visits'] for sample in batch])
    
    # 2. 결과를 담을 빈 텐서 만들기 (모두 0으로 초기화)
    # shape: (배치크기, 최대방문길이, 전체코드개수)
    batch_size = len(batch)
    
    # 진단(C), 시술(P), 약물(D) 각각에 대해 멀티-핫 벡터 생성
    # vocabs['diag']는 딕셔너리이므로 len()으로 크기를 구함
    num_diag = len(vocabs['diag'])
    num_proc = len(vocabs['proc'])
    num_drug = len(vocabs['drug'])

    # 3가지 텐서 준비 (Batch, Time, Vocab)
    tensor_diag = torch.zeros((batch_size, max_visits, num_diag))
    tensor_proc = torch.zeros((batch_size, max_visits, num_proc))
    tensor_drug = torch.zeros((batch_size, max_visits, num_drug))
    
    # 마스크(Mask): 진짜 데이터가 있는 곳은 1, 패딩인 곳은 0
    # (나중에 RNN이 "여기는 가짜니까 무시해!"라고 알 수 있게 함)
    mask = torch.zeros((batch_size, max_visits))

    # 3. 데이터 채워 넣기
    for i, sample in enumerate(batch):
        visits = sample['num_visits']
        
        # 마스크 표시 (방문한 횟수만큼 1)
        mask[i, :visits] = 1
        
        # 각 방문(t)마다 해당되는 코드 위치를 1로 변경 (Multi-hot)
        for t in range(visits):
            # 진단 코드들
            for code in sample['diagnosis'][t]:
                if code < num_diag: tensor_diag[i, t, code] = 1
            
            # 시술 코드들
            for code in sample['procedure'][t]:
                if code < num_proc: tensor_proc[i, t, code] = 1
                
            # 약물 코드들
            for code in sample['medication'][t]:
                if code < num_drug: tensor_drug[i, t, code] = 1

    return {
        'diagnosis': tensor_diag,   # 입력 1
        'procedure': tensor_proc,   # 입력 2
        'medication': tensor_drug,  # 입력 3
        'mask': mask,               # 길이 정보
        'patient_ids': [s['patient_id'] for s in batch]
    }

print("✅ collate_fn 함수 정의 완료!")

✅ collate_fn 함수 정의 완료!


In [7]:
# 1. 데이터로더 생성 (배치 크기 2로 설정해보기)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)

# 2. 딱 한 묶음(Batch)만 꺼내서 모양 확인
data_iter = iter(dataloader)
batch_data = next(data_iter)

print("=== Batch Data Shape Check ===")
print(f"1. Diagnosis Tensor: {batch_data['diagnosis'].shape}")
print(f"   (Batch Size, Max Visits, Diag Vocab Size)")
print(f"2. Procedure Tensor: {batch_data['procedure'].shape}")
print(f"3. Medication Tensor: {batch_data['medication'].shape}")
print(f"4. Mask Shape: {batch_data['mask'].shape}")
print(f"5. Patient IDs: {batch_data['patient_ids']}")

=== Batch Data Shape Check ===
1. Diagnosis Tensor: torch.Size([2, 5, 1472])
   (Batch Size, Max Visits, Diag Vocab Size)
2. Procedure Tensor: torch.Size([2, 5, 352])
3. Medication Tensor: torch.Size([2, 5, 631])
4. Mask Shape: torch.Size([2, 5])
5. Patient IDs: [10038992, 10007795]
